# ACORec meta-dev similarity/search ablation
Runs one resumable, leak-free similarity × search combination. Use `SHARD_INDEX=0..NUM_SHARDS-1`; TPOT scores only the frozen outer 20% test split.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/MothMalone/SolutionRecommendation.git'
BRANCH = 'experiment/aco-search-ablation'
SIMILARITY_VARIANT = 'rank_mse'
SEARCH_VARIANT = 'global_control'
STAGE = 'both'                 # search, tpot, both
NUM_SHARDS = 18
SHARD_INDEX = 0               # 0..17; one dataset per shard
ACO_SEEDS = [42, 43, 44]
CONFIRMATION = False          # False=TPOT 1m; True=TPOT 5m
RESUME_INPUT_DIR = None       # optional attached Kaggle output folder

REPO_DIR = Path('/kaggle/working/SolutionRecommendation')
OUTPUT_DIR = Path('/kaggle/working/acorec_meta_dev_ablation')
CACHE_DIR = Path('/kaggle/working/acorec_meta_dev_data')


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-tpot-kaggle.txt')], check=True)
if RESUME_INPUT_DIR:
    source = Path(RESUME_INPUT_DIR)
    if source.exists():
        shutil.copytree(source, OUTPUT_DIR, dirs_exist_ok=True)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True).strip())


In [ ]:
command = [
    sys.executable, str(REPO_DIR / 'scripts/run_acorec_meta_dev_ablation.py'),
    '--similarity-variant', SIMILARITY_VARIANT, '--search-variant', SEARCH_VARIANT,
    '--stage', STAGE, '--num-shards', str(NUM_SHARDS), '--shard-index', str(SHARD_INDEX),
    '--aco-seeds', *[str(seed) for seed in ACO_SEEDS],
    '--output-dir', str(OUTPUT_DIR), '--cache-dir', str(CACHE_DIR),
]
if CONFIRMATION:
    command.append('--confirmation')
env = os.environ.copy()
env.update({'PYTHONUNBUFFERED':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','OPENBLAS_NUM_THREADS':'1','NUMEXPR_NUM_THREADS':'1'})
print(' '.join(command))
subprocess.run(command, cwd=REPO_DIR, env=env, check=True)


In [ ]:
combo = f'sim={SIMILARITY_VARIANT}__search={SEARCH_VARIANT}'
archive = shutil.make_archive(str(Path('/kaggle/working') / f'{combo}_shard_{SHARD_INDEX:02d}'), 'gztar', root_dir=OUTPUT_DIR / combo)
print('Archive:', archive)
